In [ ]:

from pathlib import Path
import re
from collections import defaultdict
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
PREP_ROOT = Path(r"D:\Amrita\Final Year Project\fyp 2026\Dataset\flair_sen_tifs_preproccessed")  # your resampled inputs (set to bicubic root if needed)
OUT_ROOT  = PREP_ROOT                             # keep same root for outputs
RUN_NORMALIZATION = False                         # set False to skip normalized/
PREVIEW_PER_ZONE = 1                              # one preview per zone

# Alignment resampling kernel inside compositing (only used if grids differ):
ALIGN_RESAMPLING = "bilinear"   # choose "bilinear" or "cubic"

# (Optional) restrict zones for normalization stats (keep empty to use all zones)
NORMALIZATION_ZONE_WHITELIST = []      # e.g., ["D004_2021","D014_2020"]

# =========================
# CONSTANTS / HELPERS
# =========================
BAND = {"B02":1,"B03":2,"B04":3,"B05":4,"B06":5,"B07":6,"B08":7,"B8A":8,"B11":9,"B12":10}

DATE_PATS = [
    re.compile(r'(?P<y>\d{4})[-_ ](?P<m>\d{2})[-_ ](?P<d>\d{2})'),  # 2021-01-03 / 2021_01_03
    re.compile(r'(?P<y>\d{4})(?P<m>\d{2})(?P<d>\d{2})'),            # 20210103
]

def month_key_from_path(p: Path) -> str:
    # search folders (right-to-left)
    for part in reversed(p.parts):
        for pat in DATE_PATS:
            m = pat.search(part)
            if m:
                return f"{m.group('y')}-{m.group('m')}"
    # fallback: search filename
    for pat in DATE_PATS:
        m = pat.search(p.name)
        if m:
            return f"{m.group('y')}-{m.group('m')}"
    return ""

def is_output_path(tif: Path) -> bool:
    # avoid feedback loops by ignoring our own outputs
    parts = set(tif.parts)
    return ("composites" in parts) or ("normalized" in parts)

def monthly_files_by_zone(prep_root: Path):
    """Return dict: zone -> month -> list[tif_path] (inputs only, excluding outputs)."""
    by_zone_month = defaultdict(lambda: defaultdict(list))
    for zone_dir in sorted([p for p in prep_root.iterdir() if p.is_dir()]):
        for tif in zone_dir.rglob("*.tif"):
            if is_output_path(tif):
                continue
            mk = month_key_from_path(tif)
            if mk:
                by_zone_month[zone_dir.name][mk].append(tif)
    return by_zone_month

def build_ref_profile(tif_path: Path):
    """Reference grid from the first scene of the month."""
    with rasterio.open(tif_path) as ds:
        ref = ds.profile.copy()
        ref["dtype"] = "float32"
        return ref

def _to_reflectance(arr: np.ndarray) -> np.ndarray:
    """Scale DN -> reflectance (0..1) if needed."""
    finite = np.isfinite(arr)
    if not finite.any():
        return arr.astype(np.float32)
    vmax = np.nanpercentile(arr[finite], 99.9)
    if vmax > 2.0:  # typical DN near ~10000
        arr = arr / 10000.0
    return arr.astype(np.float32)

# --- NEW: counters for alignment stats ---
aligned_scenes = 0   # scenes reprojected to reference grid
as_is_scenes   = 0   # scenes already on the same grid (no reprojection)

def read_maybe_aligned(tif_path: Path, ref_profile: dict) -> np.ndarray:
    """
    Read multi-band tif -> (C,H,W) float32 aligned to ref grid.
    - Converts nodata to NaNs
    - Converts to reflectance (0..1) if needed
    - Reprojects only if grid differs (avoids double blur)
    """
    global aligned_scenes, as_is_scenes

    with rasterio.open(tif_path) as src:
        C = src.count
        H, W = ref_profile["height"], ref_profile["width"]

        same_grid = (
            (src.crs == ref_profile["crs"]) and
            (src.transform == ref_profile["transform"]) and
            (src.height == H) and (src.width == W)
        )

        out = np.full((C, H, W), np.nan, dtype=np.float32)
        rs = Resampling.cubic if ALIGN_RESAMPLING.lower() == "cubic" else Resampling.bilinear

        for i in range(1, C+1):
            band = src.read(i).astype(np.float32)

            # nodata -> NaN
            nd = None
            try:
                nd = src.nodatavals[i-1]
            except Exception:
                nd = None
            if nd is not None:
                band = np.where(band == nd, np.nan, band)

            # scale to reflectance BEFORE compositing
            band = _to_reflectance(band)

            if same_grid:
                out[i-1] = band
            else:
                dest = np.zeros((H, W), dtype=np.float32)
                reproject(
                    source=band,
                    destination=dest,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_profile["transform"],
                    dst_crs=ref_profile["crs"],
                    dst_height=H,
                    dst_width=W,
                    resampling=rs,
                )
                out[i-1] = dest

        # clamp for safety
        out = np.clip(out, 0.0, 1.0)

        # --- update counters ---
        if same_grid:
            as_is_scenes += 1
        else:
            aligned_scenes += 1

        return out

def stack_read_aligned(tif_list):
    """Align all tifs to the first one's grid and stack -> (N,C,H,W), ref_profile."""
    if not tif_list:
        return None, None
    ref = build_ref_profile(tif_list[0])
    arrays = []
    for fp in tif_list:
        try:
            arrays.append(read_maybe_aligned(fp, ref))
        except Exception as e:
            print(f"      ! drop {fp.name}: {e}")
    if not arrays:
        return None, None
    return np.stack(arrays, axis=0), ref  # (N,C,H,W)

def composite_median(stack: np.ndarray) -> np.ndarray:
    """(N,C,H,W) -> (C,H,W) nanmedian."""
    return np.nanmedian(stack, axis=0).astype(np.float32)

def write_geotiff(out_path: Path, array: np.ndarray, ref_profile: dict):
    """Save (C,H,W) array as GeoTIFF with ref_profile metadata."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    prof = ref_profile.copy()
    prof.update({
        "count": int(array.shape[0]),
        "height": int(array.shape[1]),
        "width":  int(array.shape[2]),
        "dtype":  "float32",
        "compress": "deflate"
    })
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(array)

def aggregate_zone_month(zone: str, month: str, files):
    """Align → reflectance → median composite → write to composites/."""
    stack, ref = stack_read_aligned(sorted(files))
    if stack is None:
        return None
    comp = composite_median(stack)     # (C,H,W) reflectance already
    out_path = OUT_ROOT / zone / "composites" / f"{zone}_{month}.tif"
    write_geotiff(out_path, comp, ref)
    return out_path, comp  # return the array for preview

# ---------- preview helpers ----------
def _stretch_99(rgb: np.ndarray) -> np.ndarray:
    finite = np.isfinite(rgb)
    if not finite.any():
        return rgb
    p99 = np.nanpercentile(rgb[finite], 99)
    p99 = float(p99) if p99 > 0 else 1.0
    return np.clip(rgb / p99, 0, 1)

def show_rgb(comp_arr: np.ndarray, title: str, bands=(3,2,1)):
    """Show natural color using B04,B03,B02 by default (1-based indices)."""
    C = comp_arr.shape[0]
    if max(bands) > C:
        print(f"[preview] Not enough bands to render RGB (have {C}, need {max(bands)})."); return
    rgb = np.stack([comp_arr[b-1] for b in bands], axis=-1).astype(np.float32)
    rgb = _stretch_99(rgb)
    plt.figure(figsize=(6,6)); plt.imshow(rgb); plt.title(title); plt.axis("off"); plt.show()

# =========================
# PASS 1: Build composites (one preview per zone)
# =========================
by_zone_month = monthly_files_by_zone(PREP_ROOT)
print(f"Zones found: {len(by_zone_month)}")

for zone, months in by_zone_month.items():
    print(f"\n=== {zone} ===")
    previewed = 0  # reset per zone
    for month, files in sorted(months.items()):
        files = sorted(files)
        print(f"  {month}: {len(files)} scenes")
        try:
            out_info = aggregate_zone_month(zone, month, files)
            if out_info:
                out_path, comp_arr = out_info
                print(f"     → saved: {out_path.name}  shape={comp_arr.shape}")
                if previewed < PREVIEW_PER_ZONE:
                    show_rgb(comp_arr, f"{zone}  {month}  (RGB B04-B03-B02)",
                             bands=(BAND['B04'], BAND['B03'], BAND['B02']))
                    previewed += 1
        except Exception as e:
            print(f"     ! failed: {e}")

# --- NEW: alignment summary after pass 1 ---
print("\n=== Alignment summary ===")
print(f"  Scenes read as-is (same grid): {as_is_scenes}")
print(f"  Scenes reprojected/aligned   : {aligned_scenes}")

# =========================
# PASS 2: Global normalization over composites (optional)
# =========================
if RUN_NORMALIZATION:
    comp_by_zone = defaultdict(list)
    for zone_dir in sorted([p for p in PREP_ROOT.iterdir() if p.is_dir()]):
        if NORMALIZATION_ZONE_WHITELIST and (zone_dir.name not in NORMALIZATION_ZONE_WHITELIST):
            continue
        comp_dir = zone_dir / "composites"
        if comp_dir.is_dir():
            for tif in comp_dir.glob("*.tif"):
                comp_by_zone[zone_dir.name].append(tif)

    sums = sumsq = counts = None
    total_files = sum(len(v) for v in comp_by_zone.values())
    print(f"\n[Normalization] Files considered for stats: {total_files}")

    for zone, files in comp_by_zone.items():
        for fp in files:
            try:
                with rasterio.open(fp) as ds:
                    arr = ds.read().astype(np.float32)  # (C,H,W)
                finite = np.isfinite(arr)
                if sums is None:
                    C = arr.shape[0]
                    sums  = np.zeros(C, dtype=np.float64)
                    sumsq = np.zeros(C, dtype=np.float64)
                    counts = np.zeros(C, dtype=np.int64)
                for b in range(arr.shape[0]):
                    v = arr[b][finite[b]]
                    if v.size:
                        sums[b]  += np.sum(v, dtype=np.float64)
                        sumsq[b] += np.sum(v * v, dtype=np.float64)
                        counts[b] += v.size
            except Exception as e:
                print(f"  ! skip {fp.name}: {e}")

    if (counts is None) or (counts.sum() == 0):
        print("[Normalization] No valid data found; skipping normalization.")
    else:
        means = sums / np.maximum(counts, 1)
        vars_ = (sumsq / np.maximum(counts, 1)) - (means * means)
        stds  = np.sqrt(np.maximum(vars_, 1e-12)).astype(np.float32)
        means = means.astype(np.float32)

        print("[Normalization] Per-band means:", np.round(means, 6))
        print("[Normalization] Per-band stds :", np.round(stds, 6))

        for zone, files in comp_by_zone.items():
            out_dir = PREP_ROOT / zone / "normalized"
            out_dir.mkdir(parents=True, exist_ok=True)
            for fp in files:
                try:
                    with rasterio.open(fp) as ds:
                        comp = ds.read().astype(np.float32)
                        ref = ds.profile.copy(); ref["dtype"] = "float32"
                    for b in range(comp.shape[0]):
                        comp[b] = (comp[b] - means[b]) / stds[b]
                    out_path = out_dir / fp.name
                    write_geotiff(out_path, comp, ref)
                    print(f"     → normalized: {zone}\\normalized\\{fp.name}")
                except Exception as e:
                    print(f"  ! normalize fail {fp.name}: {e}")


In [ ]:

from pathlib import Path
import json
import re
import numpy as np
import rasterio
from tqdm import tqdm

# ========= CONFIG =========
ROOT = Path(r"D:\Amrita\Final Year Project\fyp 2026\Dataset\flair_sen_tifs_preproccessed")   # where <zone>\composites\*.tif live

# Option A: explicitly list training zones (folder names under ROOT)
TRAIN_ZONES = [
    # e.g. "D004_2021", "D006_2020"
]

# Option B: or select by year suffix found in zone name (e.g., *_2019, *_2020)
TRAIN_YEARS = {"2018", "2019"}   # <- set as needed; empty set = ignore year filter

# If both A and B are provided, a zone is in TRAIN if it matches either.

# ========= HELPERS =========
def iter_composites(root: Path, zones_filter=None):
    """
    Yield (zone, comp_path). If zones_filter is provided (set of zone names),
    only those zones are traversed.
    """
    for zone_dir in root.iterdir():
        if not zone_dir.is_dir():
            continue
        zname = zone_dir.name
        if zones_filter is not None and zname not in zones_filter:
            continue
        comp_dir = zone_dir / "composites"
        if not comp_dir.exists():
            continue
        for tif in comp_dir.glob("*.tif"):
            yield zname, tif

def zone_year(zname: str):
    # expects zone format like D006_2020 -> returns "2020"; else None
    m = re.search(r"_(\d{4})$", zname)
    return m.group(1) if m else None

def choose_train_zones(root: Path) -> set:
    zones = {p.name for p in root.iterdir() if p.is_dir()}
    chosen = set()
    if TRAIN_ZONES:
        chosen |= set(z for z in TRAIN_ZONES if z in zones)
    if TRAIN_YEARS:
        chosen |= {z for z in zones if zone_year(z) in TRAIN_YEARS}
    return chosen

def detect_and_scale_to_reflectance(arr: np.ndarray):
    """
    If data look like DN (max > 2), divide by 10000 to get reflectance in 0–1.
    """
    finite = np.isfinite(arr)
    if not finite.any():
        return arr
    vmax = np.nanpercentile(arr[finite], 99.9)
    return (arr / 10000.0).astype(np.float32) if vmax > 2.0 else arr.astype(np.float32)

def batch_update_mean_M2(n, mean, M2, x):
    """
    Combine running stats (n, mean, M2) with a new batch x (1D np.array, finite only).
    Returns updated (n, mean, M2). Works per-band when called for each band.
    """
    if x.size == 0:
        return n, mean, M2
    k = x.size
    mk = x.mean()
    # sum of squared deviations from batch mean
    M2k = np.sum((x - mk) ** 2)
    if n == 0:
        return k, mk, M2k
    delta = mk - mean
    n_new = n + k
    mean_new = mean + delta * (k / n_new)
    M2_new = M2 + M2k + delta**2 * (n * k / n_new)
    return n_new, mean_new, M2_new

# ========= PASS 1: compute global mean/std on TRAIN ONLY =========
train_zones = choose_train_zones(ROOT)
if not train_zones:
    raise ValueError("No training zones selected. Set TRAIN_ZONES and/or TRAIN_YEARS.")

print("Training zones:", sorted(train_zones))

n = None
mean = None
M2 = None

for zone, comp in tqdm(list(iter_composites(ROOT, zones_filter=train_zones)), desc="Pass1 (stats)"):
    with rasterio.open(comp) as ds:
        data = ds.read().astype(np.float32)    # (C,H,W)
        data = detect_and_scale_to_reflectance(data)
        C = data.shape[0]
        if mean is None:
            n = np.zeros(C, dtype=np.int64)
            mean = np.zeros(C, dtype=np.float64)
            M2 = np.zeros(C, dtype=np.float64)

        for b in range(C):
            band = data[b].ravel()
            msk = np.isfinite(band)
            x = band[msk].astype(np.float64)
            n[b], mean[b], M2[b] = batch_update_mean_M2(n[b], mean[b], M2[b], x)

std = np.sqrt(np.where(n > 1, M2 / (n - 1), 0.0)).astype(np.float32)
mean = mean.astype(np.float32)

print("Per-band mean:", np.round(mean, 6))
print("Per-band std :", np.round(std, 6))

# Save stats
stats = {
    "train_zones": sorted(train_zones),
    "mean": mean.tolist(),
    "std": std.tolist(),
}
stats_path = ROOT / "global_normalization_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print("Saved stats ->", stats_path)

# ========= PASS 2: apply to ALL ZONES (write normalized copies) =========
def normalize_and_write(src_path: Path, dst_path: Path, mean: np.ndarray, std: np.ndarray):
    with rasterio.open(src_path) as ds:
        arr = ds.read().astype(np.float32)          # (C,H,W)
        arr = detect_and_scale_to_reflectance(arr)  # 0–1 reflectance if needed
        eps = 1e-6
        for b in range(arr.shape[0]):
            arr[b] = (arr[b] - mean[b]) / (std[b] + eps)

        prof = ds.profile.copy()
        prof.update({"dtype": "float32", "compress": "deflate"})
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(dst_path, "w", **prof) as out:
            out.write(arr)

print("\nWriting normalized copies (all zones) …")
for zone, comp in tqdm(list(iter_composites(ROOT)), desc="Pass2 (normalize)"):
    norm_dir = ROOT / zone / "normalized"
    out = norm_dir / comp.name  # same filename, different subfolder
    try:
        normalize_and_write(comp, out, mean, std)
    except Exception as e:
        print(f"  ! failed {comp} -> {e}")

print("\n✅ Done. Normalized composites are in:  D:\\fyp\\flair2_preprocessed\\<zone>\\normalized\\")
    
